# No tag pruning or preprocessing

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import re
import numpy as np
import networkx as nx

pd.set_option('display.max_colwidth', None)
np.set_printoptions(legacy='1.25')

ROOT = Path.cwd()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent

DATA = ROOT / "data" / "prb_articles_labeled_Jan2016-Jun2026_duplicate-free.json"
DATA_SAMPLE = ROOT / "data" / "prb_articles_labeled_Jan2016-Jun2026_duplicate-free_sample.json"

#CLEANED_DATA is output of prep_data() from preprocessing.py
CLEANED_DATA  = ROOT / "data" / "cleaned_data.json"
CLEANED_DATA_SAMPLE  = ROOT / "data" / "cleaned_data_sample.json"


MODULE_PATH = str(ROOT / "src")
if MODULE_PATH not in sys.path:
    sys.path.append(MODULE_PATH)



data_to_load = CLEANED_DATA if CLEANED_DATA.exists() else CLEANED_DATA_SAMPLE
df = pd.read_json(data_to_load)

print(f"Loaded data from {data_to_load.name}")
df.info()

Loaded data from prb_articles_labeled_Jan2016-Jun2026_duplicate-free.json
<class 'pandas.DataFrame'>
RangeIndex: 50398 entries, 0 to 50397
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   doi          50398 non-null  str           
 1   title        50398 non-null  str           
 2   abstract     50398 non-null  str           
 3   physh        50398 non-null  object        
 4   date         50398 non-null  datetime64[us]
 5   articleType  50398 non-null  str           
 6   physh_names  50398 non-null  object        
dtypes: datetime64[us](1), object(2), str(4)
memory usage: 2.7+ MB


In [ ]:
#For loading NetworkX PhySH graph
from data.preprocessing import concept_hierarchy_graph

graph_hierarchy = concept_hierarchy_graph()
label_naming = nx.get_node_attributes(graph_hierarchy,'label')
labeled_hierarchy = nx.relabel_nodes(graph_hierarchy, label_naming, copy=True)

## Preprocessing input

In [ ]:
from data.preprocessing import prep_data

#Cleans up abstract data
df = prep_data(df)
df.head()

In [ ]:
#df.to_json(ROOT / "data" / 'cleaned_data.json', orient='records',compression="infer",date_format='iso')

In [ ]:
df['articleType'].value_counts()

## Lemmatizing and Vectorizing

In [4]:
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from nltk.corpus import stopwords

WNlemma = nltk.WordNetLemmatizer()


In [5]:
stopwords = set(stopwords.words('english'))
remove_punctuations = re.compile(r'[\-\(\)\{\}\[\]|@,;]')
#Lemmatize
df['abstract'] = df["abstract"].apply(lambda x: ' '.join(WNlemma.lemmatize(word) for word in x.split() if word not in stopwords))
df['abstract'] = df["abstract"].apply(lambda x: remove_punctuations.sub(' ', x))

In [6]:
df['abstract'] = df["abstract"].apply(lambda x: x.lower())
df['title'] = df["title"].apply(lambda x: x.lower())

In [7]:
#this cell takes 2m25s to run for ngram_range(1,3)
vect_tfidf = TfidfVectorizer(min_df=10,ngram_range=(1,1),stop_words=list(stopwords))


In [8]:
abstracts_transformed=vect_tfidf.fit_transform(df['abstract'])
feature_freq_df = pd.DataFrame(abstracts_transformed.toarray().sum(axis=0))
feature_freq_df.rename({idx: val for idx,val in enumerate(vect_tfidf.get_feature_names_out())},inplace=True,axis=0)
feature_freq_df.sort_values(by=0,ascending=False,inplace=True)
feature_freq_df.shape

(11759, 1)

In [9]:
feature_freq_df.tail()

,0
begun,1.266036
6nm,1.263624
maes,1.258751
slopes,1.175969
shortened,1.153376


In [ ]:
#this cell takes 38s to run for ngram_range(1,3)
#vect_count = CountVectorizer(min_df=10,ngram_range=(1,1),stop_words=list(stopwords))
#abstracts_transformed=vect_count.fit_transform(df['Abstracts'])

## Training

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from napkinxc.models import PLT
from napkinxc.metrics import precision_at_k
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MultiLabelBinarizer



In [9]:
target = 'physh_names'
features = ['title','abstract']
X = df[features]
y = df[target]

In [10]:
# Splitting the dataset into train and test sets for single label
X_train_preprune, X_test_preprune, y_train_preprune, y_test_preprune = train_test_split(X.squeeze(), y, test_size=0.2, random_state=0)


## Multinomial NB

In [11]:
mlb = MultiLabelBinarizer()
y_train_binarized_preprune = mlb.fit_transform(y_train_preprune)
y_test_binarized_preprune = mlb.transform(y_test_preprune)
len(mlb.classes_)

/home/rustycutlery/cppexampl/pys/PhySH-Tank/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['A ≥ 220', 'Accelerator subsystems', 'Accelerators & storage rings', 'Astronomical black holes', 'Beam code development & simulation techniques', 'Beam polarization', 'Beta decay', 'Biomolecular dynamics', 'Biomolecular structure', 'Block copolymers', 'Cancer', 'Carbohydrates', 'Cavity magnonics', 'Cell membrane', 'Chalcogens', 'Charge migration', 'Chemical hydrogen storage', 'Cold working', 'Conformal foams', 'Diversity', 'Efimov states', 'Exclusion processes', 'Fluorescence recovery after photobleaching', 'Fluorescent biomolecules', 'Foams', 'Form factors', 'Fractional Brownian motion', 'Gamma ray spectroscopy', 'Gravitational wave detectors', 'Helicity', 'Hydrogen production', 'Infrastructures & instrumentation', 'Interconnected & interdependent networks', 'Interfacial flows', 'Isotope shift', 'Kelvin, coastal, & edge waves', 'Kelvin-He

2348

In [12]:
#This cell takes 1m46s to run
X_train_transformed_preprune = vect_tfidf.fit_transform(X_train_preprune['title'] + " " + X_train_preprune['abstract'])
X_test_transformed_preprune = vect_tfidf.transform(X_test_preprune['abstract'])
model_multinomialNB_preprune = MultiOutputClassifier(MultinomialNB(alpha=0.01))
model_multinomialNB_preprune.fit(X_train_transformed_preprune,y_train_binarized_preprune)

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.A :term:`predict_proba` method will be exposed only if `estimator` implementsit.,MultinomialNB(alpha=0.01)
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary <n_jobs>` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",None
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)Class labels.",list,"[array([0, 1]), array([0, 1]), array([0, 1]), array([0, 1]), ...]"
estimators_ estimators_: list of ``n_output`` estimatorsEstimators used for predictions.,list,"[MultinomialNB(alpha=0.01), MultinomialNB(alpha=0.01), MultinomialNB(alpha=0.01), MultinomialNB(alpha=0.01), ...]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying `estimator` exposes such an attribute when fit... versionadded:: 0.24,int,10807
,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",0.01
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [13]:
prediction_multinomialNB_preprune = model_multinomialNB_preprune.predict(X_test_transformed_preprune)
prediction_probs_multinomialNB_preprune = model_multinomialNB_preprune.predict_proba(X_test_transformed_preprune)

In [20]:
#prediction for a given abstract
input_string = "Signatures of fractionalization in the optical phonons of the hyperhoneycomb Kitaev magnet " + "In this study, we propose that the signatures of spin fractionalization in quantum magnets can be identified through a detailed analysis of the temperature dependence of the asymmetric Fano lineshape of optical phonons overlapping with a continuum of spin excitations. We focus on the hyperhoneycomb magnet 𝛽−Li2⁢IrO3, a promising candidate for being in proximity to a three-dimensional Kitaev quantum spin liquid. The Raman response in 𝛽−Li2⁢IrO3 notably displays a distinctive asymmetric Fano lineshape in the 24 meV Raman-active optical phonon. This asymmetry arises from the interaction between the discrete phonon mode and the spin excitation continuum, which could be fractionalized if the material is indeed near a quantum spin-liquid phase. Our theoretical model considers the coupling of this optical phonon to Majorana fermions in the Kitaev model on the hyperhoneycomb lattice. Our findings reveal that the temperature-dependent Fano lineshape is consistent with the fractionalization of spins into Majorana fermions and ℤ⁢2 fluxes."
input_transformed = vect_tfidf.transform([input_string])
input_prediction=model_multinomialNB_preprune.predict(input_transformed)
mlb.inverse_transform(input_prediction)

print(input_prediction)
print(max(model_multinomialNB_preprune.estimators_[0].predict_proba(input_transformed)[0]))

probs = np.array([est.predict_proba(input_transformed)[0][1] for est in model_multinomialNB_preprune.estimators_])
predicted_labels = mlb.classes_[probs > 0.2]
print(predicted_labels)

[[0 0 0 ... 0 0 0]]
0.9999677690752826
['Kitaev model' 'Quantum spin liquid' 'Raman spectroscopy'
 'Spin-phonon coupling']


## napkinXC

In [22]:
# Load Probabilistic Label Tree if already exists, train if doesn't
napkinxc_preprune_model_path = Path("physh_plt_model_preprune")

if (napkinxc_preprune_model_path / "weights.bin").exists():
    model_napkinXC_preprune = PLT(str(napkinxc_preprune_model_path))

else:
    model_napkinXC_preprune = PLT(str(napkinxc_preprune_model_path))
    model_napkinXC_preprune.fit(X_train_transformed_preprune, y_train_binarized_preprune)



In [ ]:
#Invoke model to make predictions on the test set
prediction_napkinXC_preprune = model_napkinXC_preprune.predict(X_test_transformed_preprune, top_k=5)

# Assign contiguous integer IDs to NetworkX nodes
for idx, node in enumerate(labeled_hierarchy.nodes()):
    labeled_hierarchy.nodes[node]['napkin_id'] = idx

# Reverse mapping for prediction decoding
id_to_tag = {data['napkin_id']: node for node, data in labeled_hierarchy.nodes(data=True)}

# Predictions DOIs mapped back to names
predicted_tags_napkinXC_preprune = [[id_to_tag[label_id] for label_id in sample] for sample in predictions_napkinXC_preprune]


## Model Evaulation

In [14]:
from sklearn.metrics import classification_report,multilabel_confusion_matrix
import sklearn.metrics as skm
import matplotlib.pyplot as plt


In [42]:
renaming_classes = dict(zip(np.arange(len(mlb.classes_)).astype(str),mlb.classes_))
report_df_multinomialNB_preprune=pd.DataFrame(classification_report(y_test_binarized_preprune,
                      prediction_multinomialNB_preprune,
                      output_dict=True,
                      zero_division=1.0)).rename(columns=renaming_classes).transpose().sort_values(by=['support'])

In [43]:
report_df_multinomialNB_preprune.tail()

,precision,recall,f1-score,support
Electronic structure,0.498969,0.241758,0.325707,1001.0
samples avg,0.755043,0.128999,0.150622,58224.0
weighted avg,0.534285,0.117855,0.161708,58224.0
macro avg,0.854735,0.247931,0.259870,58224.0
micro avg,0.444747,0.117855,0.186333,58224.0


In [44]:
confusion_matrices = dict(zip(mlb.classes_,multilabel_confusion_matrix(y_test_binarized_preprune,prediction_multinomialNB_preprune)))


In [45]:
confusion_matrices.get('Adiabatic demagnetization')

array([[10080,     0],
       [    0,     0]])

### Precision@5 Calculation for MultinomialNB

In [113]:
#k=5
prob_matrix_multinomialNB = np.array(prediction_probs_multinomialNB_preprune)[:, :, 1].T
[sum([y_test_binarized_preprune[idx][[np.argsort(abstract_entry)[-k:]]].mean()  for idx,abstract_entry in enumerate(prob_matrix_multinomialNB)])/y_test_preprune.shape[0] for k in range(1,6)]

[0.5070436507936508,
 0.4394345238095238,
 0.39603174603175556,
 0.35925099206349204,
 0.3327182539682502]

In [ ]:
#semantic similarity in label space should be explored

### Precision@5 for napkinXC

In [ ]:
print(precision_at_k(y_test_binarized_preprune, prediction_napkinXC_preprune, k=5)) 


[0.56200397 0.48685516 0.43511905 0.39637897 0.36603175]


# Dropping tags with frequency <50

In [87]:
df_label_counts=df['physh_names'].explode().value_counts().to_frame(name='counts')
df_label_counts.sort_values(by='counts', ascending=True)
low_frequency_tags50 = list(df_label_counts[df_label_counts['counts']<51].index)

In [88]:
#Number of tags with frequency less than 50
print(len(low_frequency_tags50))

1556


In [89]:
df_pruned50 = df.copy()
df_pruned50['physh_names']=df_pruned50['physh_names'].apply( lambda x : [item for item in x if item not in low_frequency_tags50])

## Training

In [90]:
target = 'physh_names'
features = ['title','abstract']
X = df_pruned50[features]
y = df_pruned50[target]

# Splitting the dataset into train and test sets for single label
X_train_postprune50, X_test_postprune50, y_train_postprune50, y_test_postprune50 = train_test_split(X.squeeze(), y, test_size=0.2, random_state=0)


## Multinomial NB

In [91]:
mlb = MultiLabelBinarizer()
y_train_binarized_postprune50 = mlb.fit_transform(y_train_postprune50)
y_test_binarized_postprune50 = mlb.transform(y_test_postprune50)
len(mlb.classes_)

865

In [92]:
#This cell takes 1m46s to run
X_train_transformed_postprune50 = vect_tfidf.fit_transform(X_train_postprune50['title'] + " " + X_train_postprune50['abstract'])
X_test_transformed_postprune50 = vect_tfidf.transform(X_test_postprune50['abstract'])
model_multinomialNB__postprune50 = MultiOutputClassifier(MultinomialNB(alpha=0.01))
model_multinomialNB__postprune50.fit(X_train_transformed_postprune50,y_train_binarized_postprune50)

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.A :term:`predict_proba` method will be exposed only if `estimator` implementsit.,MultinomialNB(alpha=0.01)
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary <n_jobs>` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",None
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)Class labels.",list,"[array([0, 1]), array([0, 1]), array([0, 1]), array([0, 1]), ...]"
estimators_ estimators_: list of ``n_output`` estimatorsEstimators used for predictions.,list,"[MultinomialNB(alpha=0.01), MultinomialNB(alpha=0.01), MultinomialNB(alpha=0.01), MultinomialNB(alpha=0.01), ...]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying `estimator` exposes such an attribute when fit... versionadded:: 0.24,int,10807
,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",0.01
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [93]:
prediction_multinomialNB_postprune50 = model_multinomialNB__postprune50.predict(X_test_transformed_postprune50)
prediction_probs_multinomialNB_postprune50 = model_multinomialNB__postprune50.predict_proba(X_test_transformed_postprune50)

## napkinXC

In [55]:
# Load Probabilistic Label Tree if already exists, train if doesn't
napkinxc_postprune50_model_path = Path("physh_plt_model_postprune50")

if (napkinxc_postprune50_model_path / "weights.bin").exists():
    model_napkinXC_postprune50 = PLT(str(napkinxc_postprune50_model_path))

else:
    model_napkinXC_postprune50 = PLT(str(napkinxc_postprune50_model_path))
    model_napkinXC_postprune50.fit(X_train_transformed_postprune50, y_train_binarized_postprune50)



In [76]:
#Invoke model to make predictions on the test set
predictions_napkinXC_postprune50 = model_napkinXC_postprune50.predict(X_test_transformed_postprune50, top_k=5)

# Predictions DOIs mapped back to names
predicted_tags_napkinXC_postprune50 = [[id_to_tag[label_id] for label_id in sample] for sample in predictions_napkinXC_postprune50]


In [81]:
print(f"Pre pruning: {precision_at_k(y_test_binarized_preprune, predictions_napkinXC_preprune, k=5)}") 
print(f"Post pruning: {precision_at_k(y_test_binarized_postprune50, predictions_napkinXC_postprune50, k=5)}") 


Pre pruning: [0.56200397 0.48685516 0.43511905 0.39637897 0.36603175]
Post pruning: [0.55555556 0.48234127 0.43217593 0.39404762 0.36313492]


## Model Evaulation

In [ ]:
renaming_classes = dict(zip(np.arange(len(mlb.classes_)).astype(str),mlb.classes_))
report_df_postprune50=pd.DataFrame(classification_report(y_test_binarized,
                      prediction_pruned,
                      output_dict=True,
                      zero_division=1.0)).rename(columns=renaming_classes).transpose().sort_values(by=['support'])

In [112]:
prediction_probs_multinomialNB_postprune50[864][0]

array([9.99797827e-01, 2.02172886e-04])

In [91]:
#Ground-truth tag density of the test set
np.mean([len(row.nonzero()[0]) for row in y_test_binarized_postprune50])

np.float64(5.411011904761905)

In [ ]:
#Ground-truth tag density of the train set; similar to test set -> no bias
np.mean([len(row.nonzero()[0]) for row in y_train_binarized_postprune50])

np.float64(5.3752418274716005)

In [ ]:
report_df_postprune50.tail(4).style.format('{:.3f}')

In [ ]:
report_df_preprune.tail(4).style.format('{:.3f}')

In [ ]:
report_df_postprune50.columns = report_df_postprune50.columns + "_prune"

In [ ]:
pd.concat([report_df_preprune.tail(4),report_df_postprune50.tail(4)],axis=1)[['precision','precision_prune','recall','recall_prune','f1-score','f1-score_prune']].style.format('{:.3f}')

### Precision@5 Calculation for MultinomialNB

In [114]:
#k=5
# prediction_probs_multinomialNB_postprune50.shape = (n_labels=1556,n_test_size=10080,pos_neg_class=2)
prob_matrix_multinomialNB_postprune50 = np.array(prediction_probs_multinomialNB_postprune50)[:, :, 1].T
[sum([y_test_binarized_postprune50[idx][[np.argsort(abstract_entry)[-k:]]].mean()  for idx,abstract_entry in enumerate(prob_matrix_multinomialNB_postprune50)])/y_test_postprune50.shape[0] for k in range(1,6)]

[0.5071428571428571,
 0.4390376984126984,
 0.3953042328042422,
 0.3585565476190476,
 0.33160714285713905]

### Precision@5 for napkinXC

In [ ]:
print(precision_at_k(y_test_binarized_preprune, prediction_napkinXC_preprune, k=5)) 


[0.56200397 0.48685516 0.43511905 0.39637897 0.36603175]


# Insights

The classification report yields 4 metrics for each precision, recall and f1 score. To understand the scores, we first understand what each average means-
- **samples avg** : For a given sample (title + abstract text), each class (physh tag) gets a label TP, FP, TN, or FN. Using these calculate the f1 score and average over the entire dataset to get the "samples avg f1 score". This only appears for multilabel classification. 

- **weighted avg** : For a given class (physh tag), calculate the f1 score and then average over all classes weighted by the class support.

- **micro avg** : Calculate the TP, FP, TN and FN over all classes (physh tags), then use it to calculate a global "f1 score"

- **macro avg** : For each class (physh tag), calculate the f1 score and then average over all classes no matter how big or small a class is. 

Evaluating Multinomial Naive Bayes before and after rare-tag pruning reveals how class imbalance distorts global metrics in Extreme Multi-Label Classification (XMLC):
1. **Observation** : Precision stays the same for samples avg and micro avg while it goes down for weighted avg and macro avg.  
**Explanation** : This indicates the heavy class imbalance and since macro avg doesn't account for class weights, it shows the largest drop in precision upon tag pruning. Furthermore, the class imbalance implies that the rare tags occur in a small sample size. Thus, the model has individual high precision scores for the rare tags and tag pruning reduces the average precision by removing the tags that were inflating the macro precision. 

2. **Observation** : Recall goes up for all avgs except macro avg.  
**Explanation** : Even though the model's precision isn't affected that much, tag pruning allows for the model to catch more true positives by reducing the label space forces the model to concentrate predictions on frequent, well-represented tags, increasing the likelihood of correctly identifying relevant labels. Again, macro avg goes down because of heavy class imbalance. 

3. **Observation** : F1-scores follow the same trend as recall.  
**Explanation** : Since precision doesn't change as much, f1 follows the same trend as recall. 

4. **Recommendation** : Tag pruning is highly recommended.  
**Explanation** : Because recall overall increases on rare tag pruning and recall is the preferred metric to optimize in a tag recommendation system. It is preferable to suggest a marginally imprecise tag than to miss a relevant one. Furthermore, the loss in precision is an artifact of removing rare tags that were inflating macro precision, not a genuine degradation in model performance. A smaller label space also leads to faster training and inference.

_MultinomialNB serves as a good lightweight baseline for extreme multi-label classification (XMLC) benchmarking._